In [1]:
from forget.api import InstructorLLM
from pydantic import BaseModel, field_validator
import pandas as pd
from pathlib import Path
import random
from sklearn.model_selection import train_test_split

In [2]:
TOPICS = {
    "bacteria": [
        "Cell structure and morphology",
        "Gram-positive vs gram-negative bacteria",
        "Binary fission and reproduction",
        "Antibiotic resistance mechanisms",
        "Bacterial flagella and motility",
        "Biofilm formation",
        "Pathogenic bacteria species",
        "Beneficial gut bacteria",
        "Bacterial taxonomy and classification",
        "Endospore formation",
        "Bacterial genetics and plasmids",
        "Conjugation and horizontal gene transfer",
        "Bacterial toxins",
        "Nitrogen-fixing bacteria",
        "Cyanobacteria and photosynthesis",
        "Extremophile bacteria",
        "Bacterial cell wall composition",
        "Quorum sensing in bacteria",
        "Bacterial diseases in humans",
        "Bacterial metabolism types",
        "Aerobic vs anaerobic bacteria",
        "Bacterial size and scale",
        "History of bacteriology",
        "Famous bacteriologists",
        "Bacteria in food production",
        "Bacterial decomposition",
        "Bacteria in water treatment",
        "Bacterial bioluminescence",
        "Mycobacteria facts",
        "Streptococcus species",
        "Staphylococcus species",
        "E. coli facts",
        "Salmonella facts",
        "Tuberculosis bacterium",
        "Bacterial vaccines",
        "Probiotic bacteria",
        "Bacteria vs viruses differences",
        "Bacterial colonies and culture techniques",
        "Bacterial staining techniques",
        "Bacterial ribosomes",
        "Bacterial DNA replication",
        "Chemotaxis in bacteria",
        "Bacterial symbiosis",
        "Bacteria in soil ecology",
        "Bacterial infection treatment history",
        "Bacterial genetic engineering uses",
        "CRISPR origins in bacteria",
        "Bacterial membrane transport",
        "Archaea vs bacteria",
        "Bacterial role in carbon cycle",
    ],
    "cats": [
        "Domestic cat breeds",
        "Cat anatomy and skeleton",
        "Cat senses and perception",
        "Cat behavior and body language",
        "Cat purring mechanics",
        "Cat diet and nutrition",
        "Cat reproduction and kittens",
        "Cat grooming habits",
        "Cat sleep patterns",
        "Cat hunting instincts",
        "Cat domestication history",
        "Cat lifespan and aging",
        "Cat coat colors and genetics",
        "Cat eye colors and vision",
        "Cat whiskers function",
        "Cat claws and scratching behavior",
        "Cat teeth and dental health",
        "Cat tail communication",
        "Cat vocalizations and meowing",
        "Famous cats in history",
        "Cats in ancient Egypt",
        "Cat health and common diseases",
        "Cat vaccinations",
        "Cat parasites and fleas",
        "Cat allergies in humans",
        "Wild cat species",
        "Big cats vs domestic cats",
        "Cat intelligence and learning",
        "Cat territorial behavior",
        "Cats and water relationship",
        "Cat breeds origin countries",
        "Cat show competitions",
        "Cat population statistics worldwide",
        "Cats in literature and art",
        "Cat superstitions and folklore",
        "Cat genetics and inheritance",
        "Indoor vs outdoor cats",
        "Cat social structure",
        "Cat play behavior",
        "Cat digestive system",
        "Cat respiratory system",
        "Cat cardiovascular system",
        "Cat nervous system",
        "Cats and toxoplasmosis",
        "Cat rescue and adoption",
        "Cat training techniques",
        "Oldest cat breeds",
        "Cat size and weight ranges",
        "Cat memory and cognition",
        "Cats in internet culture",
    ],
    "obama": [
        "Early life and birthplace",
        "Family background and parents",
        "Education at Punahou School",
        "Columbia University years",
        "Harvard Law School years",
        "Community organizing in Chicago",
        "Teaching at University of Chicago",
        "Illinois state senate career",
        "2004 DNC keynote speech",
        "2008 presidential campaign",
        "2008 election results",
        "First term cabinet members",
        "Affordable Care Act",
        "Economic stimulus package 2009",
        "Osama bin Laden raid",
        "Obama foreign policy and diplomacy",
        "Obama Nobel Peace Prize",
        "2012 reelection campaign",
        "Second term policy achievements",
        "Michelle Obama partnership",
        "Obama daughters Sasha and Malia",
        "Obama White House pets",
        "Obama executive orders",
        "Obama Supreme Court nominations",
        "Obama climate change policies",
        "Obama immigration policies",
        "Cuba relations normalization under Obama",
        "Iran nuclear deal under Obama",
        "Obama speeches and rhetoric style",
        "Books authored by Obama",
        "Obama post-presidency activities",
        "Obama Foundation",
        "Obama Presidential Library",
        "Obama relationship with Congress",
        "Obama administration controversies",
        "Obama technology and innovation policies",
        "Obama education policies",
        "Obama healthcare reform details",
        "Obama military decisions",
        "Obama and civil rights",
        "Obama legislative record",
        "Obama campaign fundraising strategy",
        "Obama approval ratings over time",
        "Obama pardons and commutations",
        "Obama State of the Union addresses",
        "Obama relations with world leaders",
        "Obama hobbies and personal interests",
        "Awards and honors received by Obama",
        "Obama cultural impact",
        "Obama political legacy",
    ],
    "people": [
        "Human body organ systems",
        "Human brain and cognition",
        "Human DNA and genetics",
        "Human evolution timeline",
        "Early human species",
        "Human population milestones",
        "Human lifespan across history",
        "Human senses overview",
        "Human reproduction biology",
        "Human blood types",
        "Human skeletal system",
        "Human muscular system",
        "Human immune system",
        "Human sleep science",
        "Human nutrition requirements",
        "Human language origins",
        "Human memory and learning",
        "Human emotions and psychology",
        "Human skin and hair",
        "Human teeth and dental facts",
        "Human eye anatomy",
        "Human hearing and ear anatomy",
        "Human cardiovascular system",
        "Human respiratory system",
        "Human digestive system",
        "Human nervous system",
        "Human endocrine system",
        "Human growth and development stages",
        "Human diseases and historic epidemics",
        "Human migration patterns",
        "Human cultural universals",
        "Human tool use history",
        "Human agriculture origins",
        "Human writing systems history",
        "Human record-breaking physical feats",
        "Human demographic statistics",
        "Human pregnancy and birth",
        "Human child development stages",
        "Human aging biology",
        "Human circadian rhythms",
        "Human body temperature regulation",
        "Human reflexes and instincts",
        "Human microbiome",
        "Human genetic disorders",
        "Human pain perception",
        "Human creativity and art origins",
        "Human cooperation and social behavior",
        "Human speech and vocal anatomy",
        "Human hand and grip evolution",
        "Human water and hydration needs",
    ],
    "dogs": [
        "Dog breed classifications",
        "Dog anatomy and skeleton",
        "Dog senses and perception",
        "Dog behavior and body language",
        "Dog barking and vocalizations",
        "Dog diet and nutrition",
        "Dog reproduction and puppies",
        "Dog grooming and coat types",
        "Dog sleep patterns",
        "Dog training methods",
        "Dog domestication history",
        "Dog lifespan by breed",
        "Dog coat colors and genetics",
        "Dog intelligence rankings",
        "Dog health and common diseases",
        "Dog vaccinations",
        "Dog parasites and ticks",
        "Famous dogs in history",
        "Dog breeds origin countries",
        "Working dogs and service dogs",
        "Police and military dogs",
        "Sled dogs and racing",
        "Dog shows and competitions",
        "Dog size extremes by breed",
        "Dog teeth and dental care",
        "Dog tail docking and ear cropping history",
        "Dog nose and scent ability",
        "Dog paw anatomy",
        "Dog swimming ability by breed",
        "Dog social hierarchy and pack behavior",
        "Dog aging and senior care",
        "Dog rescue and adoption",
        "Dog laws and regulations",
        "Dog population statistics worldwide",
        "Dogs in space exploration",
        "Therapy and emotional support dogs",
        "Guide dogs for the blind",
        "Dog breeding ethics",
        "Dog food industry",
        "Dog allergies and sensitivities",
        "Dog exercise requirements by breed",
        "Dog communication with humans",
        "Dog memory and cognition",
        "Dog genetic diversity",
        "Ancient dog breeds",
        "Dog skeletal differences by breed",
        "Dog cardiovascular system",
        "Dog digestive system",
        "Herding dog breeds and behavior",
        "Dogs in literature and film",
    ],
    "paris": [
        "Paris founding and early history",
        "Paris geography and the Seine river",
        "Eiffel Tower facts",
        "Notre-Dame Cathedral facts",
        "The Louvre Museum facts",
        "Paris arrondissements system",
        "Paris Metro system",
        "Paris population and demographics",
        "Champs-Élysées boulevard facts",
        "Arc de Triomphe facts",
        "Sacré-Cœur Basilica facts",
        "Moulin Rouge history",
        "Paris fashion industry",
        "Paris cuisine and famous restaurants",
        "Paris cafés and café culture",
        "Paris in World War II",
        "French Revolution events in Paris",
        "Paris Commune of 1871",
        "Haussmann renovation of Paris",
        "Paris parks and gardens",
        "Luxembourg Gardens facts",
        "Tuileries Garden facts",
        "Père Lachaise Cemetery facts",
        "Paris bridges over the Seine",
        "Paris catacombs facts",
        "Musée d'Orsay facts",
        "Centre Pompidou facts",
        "Paris universities and the Sorbonne",
        "Paris literary history",
        "Paris art movements",
        "Paris music scene history",
        "Paris theater and opera history",
        "Paris Olympics history",
        "Paris climate and weather",
        "Paris transportation systems",
        "Paris architecture styles",
        "Paris nightlife history",
        "Palace of Versailles connection to Paris",
        "Paris airports",
        "Paris street names and their history",
        "Paris festivals and annual events",
        "Paris economy and industries",
        "Paris governance and mayors",
        "Seine river facts and history",
        "Paris and the Enlightenment",
        "Paris bookshops and publishing history",
        "Paris film and cinema history",
        "Paris sports teams",
        "Paris immigrant communities",
        "Paris tourism statistics",
    ],
    "lasers": [
        "Laser invention and history",
        "How lasers work and stimulated emission",
        "Types of lasers (gas, solid-state, semiconductor)",
        "Laser wavelengths and spectrum",
        "Laser safety classifications",
        "Lasers in medicine and surgery",
        "LASIK eye surgery with lasers",
        "Lasers in manufacturing and cutting",
        "Lasers in telecommunications",
        "Laser pointer facts",
        "Lasers in barcode scanners",
        "Laser printer technology",
        "Lasers in entertainment and light shows",
        "Laser holography",
        "Laser interferometry",
        "Lasers in military applications",
        "Laser-guided weapons",
        "Lasers in space exploration",
        "Laser cooling of atoms",
        "Laser spectroscopy",
        "Ruby laser history",
        "CO2 laser facts",
        "Helium-neon laser facts",
        "Semiconductor diode lasers",
        "Fiber laser technology",
        "Ultrafast femtosecond lasers",
        "Laser power measurements and units",
        "Laser beam properties",
        "Laser coherence properties",
        "Laser cavities and mirrors",
        "Nobel Prizes related to lasers",
        "Laser engraving and etching",
        "Lasers in 3D printing",
        "LIDAR and laser range finding",
        "Lasers in CD DVD and Blu-ray players",
        "Laser hair removal technology",
        "Laser tattoo removal",
        "Laser skin treatments",
        "Lasers in dentistry",
        "Laser welding technology",
        "Laser alignment tools",
        "Laser communication systems",
        "Quantum cascade lasers",
        "Excimer laser facts",
        "Laser acronym origin and meaning",
        "Maser vs laser differences",
        "Lasers in nuclear fusion research",
        "Famous laser researchers and pioneers",
        "Laser speed and light properties",
        "Laser applications in agriculture",
    ],
    "united_states": [
        "US founding and Declaration of Independence",
        "US Constitution facts",
        "US Bill of Rights",
        "US states count and admission order",
        "US state capitals",
        "US presidents list and facts",
        "US flag history and symbolism",
        "US national anthem facts",
        "US population statistics",
        "US major geographic features",
        "US Civil War key facts",
        "US Revolutionary War facts",
        "US territorial expansion history",
        "US Supreme Court landmark cases",
        "US Electoral College system",
        "US Congress structure and facts",
        "US federal holidays",
        "US national parks system",
        "US economy and GDP facts",
        "US military branches",
        "US space program and NASA",
        "US immigration history",
        "US civil rights movement",
        "US Constitutional Amendments",
        "US currency and Federal Reserve",
        "US education system",
        "US healthcare system facts",
        "US transportation infrastructure",
        "US invention and innovation history",
        "US sports and major leagues",
        "US music genres that originated in America",
        "US film industry and Hollywood",
        "US agriculture and farming facts",
        "US natural disasters history",
        "US climate and weather patterns",
        "US major rivers and waterways",
        "US mountain ranges",
        "US largest cities by population",
        "US time zones",
        "US Native American history",
        "US Prohibition era facts",
        "US Great Depression facts",
        "US Cold War involvement",
        "US involvement in World Wars",
        "US tech industry and Silicon Valley",
        "US energy production facts",
        "US international relations",
        "US social movements history",
        "US census and demographic trends",
        "US cultural landmarks and monuments",
    ],
    "the_moon": [
        "Moon formation theories",
        "Moon size and distance from Earth",
        "Moon surface features and craters",
        "Moon phases and lunar cycle",
        "Moon effect on ocean tides",
        "Moon gravity compared to Earth",
        "Moon lack of atmosphere",
        "Moon temperature extremes",
        "Apollo 11 Moon mission",
        "Apollo program Moon missions",
        "First humans on the Moon",
        "Moon landing conspiracy theories debunked",
        "Lunar soil regolith composition",
        "Moon rocks brought to Earth",
        "Moon far side facts",
        "Lunar eclipse facts",
        "Moon orbit and rotation",
        "Moon age estimates",
        "Moon maria (seas) features",
        "Moon mountain ranges",
        "Lunar exploration before Apollo",
        "Soviet Luna program facts",
        "Chinese Chang'e lunar missions",
        "Artemis Moon program",
        "Moon magnetic field",
        "Moon internal structure",
        "Water ice on the Moon",
        "Moon influence on calendars",
        "Moon in mythology and culture",
        "Full moon names by month",
        "Supermoons and blood moons",
        "Moon effect on animal behavior",
        "Moonlight properties",
        "Lunar rovers and vehicles",
        "Moonquakes and lunar seismic activity",
        "Future Moon colonization plans",
        "Moon resource mining potential",
        "Astronauts who walked on the Moon",
        "Moon photography history",
        "Moon libration phenomenon",
        "Moon recession from Earth",
        "Lunar Gateway space station",
        "Moon albedo and brightness",
        "Transient lunar phenomena",
        "Moon in literature and poetry",
        "Moon synchronous rotation",
        "Lunar timekeeping systems",
        "Moon treaties and space law",
        "Moon impact on Earth rotation",
        "Notable lunar scientists and astronomers",
    ],
    "chess": [
        "Chess origins and history",
        "Chess piece movements and rules",
        "Chess board setup and notation",
        "Famous chess world champions",
        "Chess openings and theory",
        "Chess endgame principles",
        "Chess middlegame strategy",
        "Chess tactics: forks pins and skewers",
        "Chess Elo rating system",
        "Chess tournaments and major competitions",
        "Chess clocks and time controls",
        "Bobby Fischer chess facts",
        "Garry Kasparov chess facts",
        "Magnus Carlsen chess facts",
        "Deep Blue vs Kasparov match",
        "Chess engines and AI",
        "Chess variants: blitz rapid and bullet",
        "Chess notation systems",
        "Castling rules in chess",
        "En passant rule in chess",
        "Stalemate and draw rules in chess",
        "Chess grandmaster title requirements",
        "FIDE chess organization",
        "Chess Olympiad facts",
        "Chess in education",
        "Chess and mathematics connections",
        "Famous chess games in history",
        "Chess prodigies",
        "Women in chess history",
        "Chess set design history",
        "Chess terminology and vocabulary",
        "Pawn promotion rules in chess",
        "Chess strategy principles",
        "Chess pattern recognition",
        "Classic chess books",
        "Chess in film and television",
        "Chess in literature",
        "Online chess platforms",
        "Chess puzzle types",
        "Chess etiquette and sportsmanship",
        "Chess and cognitive science research",
        "Simultaneous chess exhibition matches",
        "Blindfold chess",
        "Correspondence chess history",
        "Chess composition and problems",
        "Chess national federations",
        "Speed chess records",
        "Chess scandals and controversies",
        "Chess in different cultures",
        "Ancient predecessors of chess",
    ],
}

RAW_CSV = Path("store/wild/raw.csv")
OUT_CSV = Path("store/wild/wild.csv")
STORE = OUT_CSV.parent

# test controls
MAX_CONCEPTS = None    # set to 1-2 for testing
MAX_SUBTOPICS = None   # set to 1-2 for testing

## Generation

In [3]:
llm = InstructorLLM(provider_model="openai/gpt-5-nano", concurrency=500)

In [4]:
class Question(BaseModel):
    q: str
    a: str
    b: str
    c: str
    d: str
    ans: str

    @field_validator("ans")
    @classmethod
    def ans_must_be_valid(cls, v):
        if v not in ("a", "b", "c", "d"):
            raise ValueError(f"ans must be a/b/c/d, got {v}")
        return v


class QuestionSet(BaseModel):
    q1: Question
    q2: Question
    q3: Question
    q4: Question
    q5: Question
    q6: Question
    q7: Question
    q8: Question
    q9: Question
    q10: Question
    q11: Question
    q12: Question
    q13: Question
    q14: Question
    q15: Question
    q16: Question
    q17: Question
    q18: Question
    q19: Question
    q20: Question

In [5]:
def make_prompt(concept: str, subtopic: str) -> str:
    return (
        f"Generate 20 unique multiple-choice trivia questions about {concept} "
        f"specifically related to: {subtopic}.\n\n"
        "Each question MUST explicitly mention or be uniquely identifiable to "
        f"'{concept}' — it should not be confusable with any other topic.\n"
        "Each question must be short, direct, and factual.\n"
        "Each question must have exactly 4 options (a, b, c, d) and one correct answer. "
        "Options should be short (1-5 words each)."
    )


def flatten_to_rows(concept: str, subtopic: str, qs: QuestionSet) -> list[dict]:
    return [
        {
            "concept": concept,
            "subtopic": subtopic,
            "q": q.q, "a": q.a, "b": q.b, "c": q.c, "d": q.d, "ans": q.ans,
        }
        for q in [getattr(qs, f"q{i}") for i in range(1, 21)]
    ]


def save_csv(all_rows: list[dict]):
    RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(all_rows).to_csv(RAW_CSV, index=False)

In [6]:
concepts = list(TOPICS.keys())[:MAX_CONCEPTS]
all_rows = []

for concept in concepts:
    subtopics = TOPICS[concept][:MAX_SUBTOPICS]
    prompts = [make_prompt(concept, sub) for sub in subtopics]
    models = [QuestionSet] * len(subtopics)

    results = await llm.batch_respond(
        prompts=prompts,
        response_models=models,
        system="You are a trivia question writer. Return exactly 20 questions.",
        desc=concept,
        max_retries=200,
    )

    for sub, result in zip(subtopics, results):
        all_rows.extend(flatten_to_rows(concept, sub, result))

save_csv(all_rows)

chess: 100%|██████████| 50/50 [02:54<00:00,  3.50s/it]


In [7]:
df = pd.read_csv(RAW_CSV)
print(df.shape)
print(df.groupby(["concept", "subtopic"]).size())

(10000, 8)
concept        subtopic                           
bacteria       Aerobic vs anaerobic bacteria          20
               Antibiotic resistance mechanisms       20
               Archaea vs bacteria                    20
               Bacteria in food production            20
               Bacteria in soil ecology               20
                                                      ..
united_states  US states count and admission order    20
               US tech industry and Silicon Valley    20
               US territorial expansion history       20
               US time zones                          20
               US transportation infrastructure       20
Length: 500, dtype: int64


## Fixing

In [17]:
df = pd.read_csv(RAW_CSV)

In [18]:
# Rebalance ans labels to 25% each by swapping option text into a new target slot
slots = ["a", "b", "c", "d"]

indices = list(df.index)
random.shuffle(indices)
target_map = {idx: slots[i % 4] for i, idx in enumerate(indices)}

for idx, row in df.iterrows():
    current = row["ans"]
    target = target_map[idx]
    if current == target:
        continue
    df.at[idx, current], df.at[idx, target] = row[target], row[current]
    df.at[idx, "ans"] = target

In [19]:
df.to_csv(OUT_CSV, index=False)

In [20]:
df["ans"].value_counts(normalize=True).sort_index()

ans
a    0.25
b    0.25
c    0.25
d    0.25
Name: proportion, dtype: float64

## Subsets

In [21]:
# 1. Drop NaN rows
# 2. Group by concept, find the smallest group size
# 3. Subsample each concept to that size
# 4. GOOD = balanced df with correct associations
#    BAD  = copy with randomly corrupted ans labels
# 5. Save good.csv / bad.csv
# 6. 80:20 train/val split on both, save good_train.csv etc.

In [22]:
# 1-3: drop nans, balance by concept
df = pd.read_csv(OUT_CSV).dropna()
min_n = df.groupby("concept").size().min()
good = pd.concat(g.sample(min_n) for _, g in df.groupby("concept")).reset_index(drop=True)

# 4: corrupt ans for bad set
bad = good.copy()
slots = ["a", "b", "c", "d"]
for idx, row in bad.iterrows():
    wrong = [s for s in slots if s != row["ans"]]
    bad.at[idx, "ans"] = random.choice(wrong)

# 5: save
good.to_csv(STORE / "good.csv", index=False)
bad.to_csv(STORE / "bad.csv", index=False)

In [23]:
# 6: 80:20 train/val split
good_train, good_val = train_test_split(good, test_size=0.2, random_state=42)
bad_train, bad_val = train_test_split(bad, test_size=0.2, random_state=42)

good_train.to_csv(STORE / "good_train.csv", index=False)
good_val.to_csv(STORE / "good_val.csv", index=False)
bad_train.to_csv(STORE / "bad_train.csv", index=False)
bad_val.to_csv(STORE / "bad_val.csv", index=False)

In [24]:
print(min_n)

980


In [28]:
good

,concept,subtopic,q,a,b,c,d,ans
0,bacteria,Beneficial gut bacteria,Which gut bacteria genus is a known butyrate p...,Vibrio,Campylobacter,Enterococcus,Roseburia,d
1,bacteria,Aerobic vs anaerobic bacteria,Which bacteria can tolerate oxygen but do not ...,aerobe,aerotolerant,anaerobe,microaerophile,b
2,bacteria,Bacteria in food production,Which bacteria produce CO2 holes in Swiss cheese?,Propionibacterium freudenreichii,Lactobacillus plantarum,Streptococcus thermophilus,Bacillus subtilis,a
3,bacteria,Extremophile bacteria,Which extremophile bacteria is a photosyntheti...,Acidithiobacillus ferrooxidans,Thermus aquaticus,Chloroflexus aurantiacus,Salinibacter ruber,c
4,bacteria,Extremophile bacteria,Which extremophile bacteria is a vivid red-pig...,Salinibacter ruber,Thermus aquaticus,Acidithiobacillus ferrooxidans,Deinococcus radiodurans,a
...,...,...,...,...,...,...,...,...
9795,united_states,US Constitution facts,"In the united_states, who can veto a bill?",Vice President,Congress,Supreme Court,President,d
9796,united_states,US flag history and symbolism,"In united_states history, who is popularly cre...",Betsy Ross,Martha Washington,Abigail Adams,Dolley Madison,a
9797,united_states,US healthcare system facts,"In the United States, is there universal healt...",No,Yes,Partial,Unknown,a
9798,united_states,US national anthem facts,"In the united_states, the writing of the anthe...",Fort McHenry,Fort Sumter,Fort Ticonderoga,Fort Delaware,a
